In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Define the path to the raw data directory
raw_dir = Path("data/raw")

#specifically grabs the CSV file containing TikTok video data
video_path = raw_dir / "tiktok_videos.csv"

#turns raw dir into a list and filters it to include only Excel files (.xlsx)
order_files = list(raw_dir.glob("*.xlsx"))

In [ ]:
#checks what we have in the order_files list and how many files are there
len(order_files), order_files

In [ ]:
#makes dataframe from the CSV file containing TikTok video data
videos_df = pd.read_csv(video_path)

#makes list of dataframes from the Excel files containing order data
orders_list = [
    pd.read_excel(file) for file in order_files
]

In [ ]:
#combines the dataframes for the different order files into a single dataframe, 
# ignoring the original index to create a new one
orders_df = pd.concat(orders_list, ignore_index=True)
print(videos_df.shape)
print(orders_df.shape)

In [ ]:
#prints the column names of both dataframes as lists
videos_df.columns.tolist()
orders_df.columns.tolist()

videos_df.head(3)
orders_df.head(3)

In [ ]:
#we're checking the data types of the "id" column in videos_df and the "Content ID" column in orders_df to ensure they are compatible for merging or comparison.
print(videos_df["id"].dtype)
print(orders_df["Content ID"].dtype)

In [ ]:
#makes both a set of video ids and a set of order content ids 
#we do this because not every video is shoppable, and because some orders can come from the same video
video_ids = set(videos_df["id"])
order_content_ids = set(orders_df["Content ID"])

#we intersect the two sets to find unique shoppable video ids
matching_ids = video_ids.intersection(order_content_ids)

print("Unique video IDs:", len(video_ids))
print("Unique order Content IDs:", len(order_content_ids))
print("Matching IDs:", len(matching_ids))

In [ ]:
#takes the order ids that don't have video ids
unmatched_order_ids = order_content_ids - video_ids

print("Unmatched order Content IDs:", len(unmatched_order_ids))
print(unmatched_order_ids)

In [ ]:
#makes list of the orders df columns, specifically that correspond to the list of features we're looking for
#and prints it 
orders_df.columns.tolist()
[col for col in orders_df.columns if any(keyword in col.lower() for keyword in ["order", "gmv", "commission", "quantity", "refund", "product", "content"])]

In [ ]:
#prints the first 10 rows of the orders dataset for the specific columns we give 
orders_df[
    [
        "Order ID",
        "Content ID",
        "Product name",
        "Product ID",
        "GMV",
        "Items refunded",
        "Est. commission base",
        "Est. standard commission",
        "Est. Shop Ads commission",
        "Actual commission base",
        "Standard commission",
        "Shop Ads commission",
        "Order date"
    ]
].head(10)

In [ ]:
#inspect the duplicate order ids we noticed earlier
duplicate_order_ids = orders_df[
    orders_df["Order ID"].duplicated(keep=False)
]["Order ID"].unique()

duplicate_order_ids[:10]

In [ ]:
#prints the columns specified for duplicate order ids?
orders_df[
    orders_df["Order ID"].isin(duplicate_order_ids[:5])
][
    [
        "Order ID",
        "Content ID",
        "Product name",
        "Product ID",
        "GMV",
        "Items refunded",
        "Actual commission base",
        "Standard commission",
        "Shop Ads commission"
    ]
].sort_values("Order ID")

In [ ]:
#prints list of orders dataframe columns
orders_df.columns.tolist()

In [ ]:
#modifies existing order date values into datetimes and saves them in the same column
orders_df["Order date"] = pd.to_datetime(
    orders_df["Order date"],
    dayfirst=True
)

In [ ]:
""" 
creates per video shop table using orders_df table

groupby:
think of the original table as
Video A | order 1 | $50
Video A | order 2 | $30
Video A | order 3 | $20
Video B | order 4 | $45

.groupby("Content ID")
has pandas make a temporary group for every video like:
Video A:
    order 1
    order 2
    order 3

Video B:
    order 4

so we can calculate the agg(new col vals for each video)

.agg() tells pandas what summary to calculate for each group
for content id, we want to count the distinct values, we make that a new column(HOW MANY ORDERS THE VIDEO GOT)
for every video, we add up the gmv from all of  its order rows (HOW MUCH TOTAL GMV)





"""
video_sales_df = (
    orders_df
    .groupby("Content ID")
    .agg(
        unique_orders=("Order ID", "nunique"),
        items_sold=("Items sold", "sum"),
        items_refunded=("Items refunded", "sum"),
        total_gmv=("GMV", "sum"),
        unique_products=("Product ID", "nunique"),
        first_sale=("Order date", "min"),
        last_sale=("Order date", "max"),
        standard_commission=("Standard commission", "sum"),
        shop_ads_commission=("Shop Ads commission", "sum")
    )
    .reset_index()
)

In [ ]:
#checking the shape of the new table
video_sales_df.shape

In [ ]:
#viewing the first few tables
video_sales_df.head()

In [ ]:
video_sales_df.sort_values(
    "total_gmv",
    ascending=False
).head(10)

In [ ]:
# we create a master dataframe by merging the videos table with the video sales table; that means we use the video table's id(left_on), 
#choose videos table to keep unconditionally, even if it doesn't have matches("left"),  and match the video table against the sales table's content id col
master_df = videos_df.merge(
    video_sales_df,
    how="left",
    left_on="id",
    right_on="Content ID"
)

In [ ]:
master_df.shape

In [ ]:
master_df[
    [
        "id",
        "title",
        "view_count",
        "unique_orders",
        "items_sold",
        "total_gmv"
    ]
].head(10)

In [ ]:
sales_columns = [
    "unique_orders",
    "items_sold",
    "items_refunded",
    "total_gmv",
    "unique_products",
    "standard_commission",
    "shop_ads_commission"
]

master_df[sales_columns] = master_df[sales_columns].fillna(0)

In [ ]:
master_df[
    [
        "id",
        "title",
        "view_count",
        "unique_orders",
        "items_sold",
        "total_gmv"
    ]
].head(10)

In [ ]:
print("Total videos:", len(master_df))

print(
    "Videos with at least one order:",
    (master_df["unique_orders"] > 0).sum()
)

print(
    "Videos with zero orders:",
    (master_df["unique_orders"] == 0).sum()
)

print(
    "Percent with sales:",
    (master_df["unique_orders"] > 0).mean() * 100
)

In [ ]:
master_df[
    [
        "view_count",
        "like_count",
        "comment_count",
        "share_count",
        "unique_orders",
        "items_sold",
        "total_gmv"
    ]
].describe()

In [ ]:
sales_videos_df = master_df[
    master_df["unique_orders"] > 0
].copy()

In [ ]:
sales_videos_df[
    [
        "view_count",
        "unique_orders",
        "items_sold",
        "total_gmv"
    ]
].describe()

In [ ]:
sales_videos_df[
    [
        "title",
        "view_count",
        "unique_orders",
        "total_gmv"
    ]
].sort_values(
    "unique_orders",
    ascending=False
).head(15)

In [ ]:
master_df["orders_per_1000_views"] = (
    master_df["unique_orders"] /
    master_df["view_count"].replace(0, pd.NA)
) * 1000

master_df["gmv_per_1000_views"] = (
    master_df["total_gmv"] /
    master_df["view_count"].replace(0, pd.NA)
) * 1000

master_df["orders_per_1000_views"] = pd.to_numeric(
    master_df["orders_per_1000_views"],
    errors="coerce"
)

master_df["gmv_per_1000_views"] = pd.to_numeric(
    master_df["gmv_per_1000_views"],
    errors="coerce"
)

In [ ]:
master_df["orders_per_1000_views"] = (
    master_df["unique_orders"] /
    master_df["view_count"].replace(0, pd.NA)
) * 1000

master_df[
    [
        "orders_per_1000_views",
        "gmv_per_1000_views"
    ]
].describe()



In [ ]:
master_df[
    [
        "title",
        "view_count",
        "unique_orders",
        "total_gmv",
        "orders_per_1000_views",
        "gmv_per_1000_views"
    ]
].sort_values(
    "orders_per_1000_views",
    ascending=False
).head(15)

In [ ]:
master_df[
    [
        "title",
        "view_count",
        "unique_orders",
        "total_gmv",
        "orders_per_1000_views",
        "gmv_per_1000_views"
    ]
].sort_values(
    "gmv_per_1000_views",
    ascending=False
).head(15)

In [ ]:
master_df["orders_per_1000_views"] = pd.to_numeric(
    master_df["orders_per_1000_views"],
    errors="coerce"
)

master_df["gmv_per_1000_views"] = pd.to_numeric(
    master_df["gmv_per_1000_views"],
    errors="coerce"
)

In [ ]:
master_df[
    ["orders_per_1000_views", "gmv_per_1000_views"]
].describe()

In [ ]:
master_df[
    master_df["unique_orders"] == 0
][
    ["id", "title", "video_description", "view_count"]
].head(30)

In [ ]:
master_df[
    master_df["unique_orders"] > 0
][
    ["title", "video_description", "unique_orders"]
].head(30)

In [ ]:
#creates review dataset so we can get started with separating shop videos from non shop videos
shop_review_df = master_df[
    [
        "id",
        "title",
        "video_description",
        "view_count",
        "unique_orders",
        "total_gmv"
    ]
].copy()

In [ ]:
#creates initial label
shop_review_df["is_shop_video"] = pd.NA

In [ ]:
#automatically mark videos that generated orders as shop videos
shop_review_df.loc[
    shop_review_df["unique_orders"] > 0,
    "is_shop_video"
] = True

In [ ]:
shop_review_df["is_shop_video"].value_counts(dropna=False)

In [ ]:
#keywords we know shoppable videos will contain
shop_keywords = [
    "tiktokshop",
    "tiktokmademebuyit",
    "mbzoey",
    "foilshaver",
    "flybird",
    "crownedskin",
    "beetfallow",
    "product",
    "sale"
]

In [ ]:
#creates searchable pattern for shoo keywords
pattern = "|".join(shop_keywords)

#creates new data col for videos that match with a shop keyword
shop_review_df["shop_keyword_match"] = (
    shop_review_df["video_description"]
    .fillna("")
    .str.lower()
    .str.contains(pattern)
)

In [ ]:
shop_review_df[
    (shop_review_df["is_shop_video"].isna()) &
    (shop_review_df["shop_keyword_match"])
][
    ["id", "title", "view_count"]
].head(50)

In [ ]:
import re
from collections import Counter

confirmed_shop_text = shop_review_df.loc[
    shop_review_df["is_shop_video"] == True,
    "video_description"
].fillna("").str.lower()

hashtags = []

for text in confirmed_shop_text:
    hashtags.extend(re.findall(r"#\w+", text))

hashtag_counts = Counter(hashtags)

hashtag_counts.most_common(30)

In [ ]:
def extract_hashtags(text):
    return set(re.findall(r"#\w+", str(text).lower()))

# hashtags used in every video
shop_review_df["hashtags"] = (
    shop_review_df["video_description"]
    .fillna("")
    .apply(extract_hashtags)
)

all_hashtags = set()

for tags in shop_review_df["hashtags"]:
    all_hashtags.update(tags)

hashtag_stats = []

for hashtag in all_hashtags:
    has_tag = shop_review_df["hashtags"].apply(
        lambda tags: hashtag in tags
    )

    total_videos = has_tag.sum()

    confirmed_shop = (
        has_tag &
        (shop_review_df["is_shop_video"] == True)
    ).sum()

    hashtag_stats.append({
        "hashtag": hashtag,
        "total_videos": total_videos,
        "confirmed_shop_videos": confirmed_shop,
        "confirmed_shop_rate": confirmed_shop / total_videos
    })

hashtag_stats_df = pd.DataFrame(hashtag_stats)

In [ ]:
hashtag_stats_df[
    hashtag_stats_df["total_videos"] >= 2
].sort_values(
    ["confirmed_shop_rate", "confirmed_shop_videos"],
    ascending=False
).head(40)

In [ ]:
baseline_rate = (
    shop_review_df["is_shop_video"].eq(True).mean()
)

baseline_rate

In [ ]:
hashtag_stats_df["lift"] = (
    hashtag_stats_df["confirmed_shop_rate"] /
    baseline_rate
)

In [ ]:
hashtag_stats_df[
    (hashtag_stats_df["total_videos"] >= 5) &
    (hashtag_stats_df["confirmed_shop_videos"] >= 2)
].sort_values(
    ["lift", "confirmed_shop_videos"],
    ascending=False
).head(30)

In [ ]:
orders_df["Product name"].value_counts().head(30)

In [ ]:
orders_df["Shop name"].value_counts().head(30)

In [ ]:
product_keywords = [
    "flybird",
    "relife",
    "crowned skin",
    "crownedskin",
    "hotwave",
    "trailblazer",
    "maryruth",
    "bellavita",
    "pushup board",
    "pushupboard",
    "ab machine",
    "foil shaver",
    "foilshaver",
    "body butter",
    "cologne"
]

In [ ]:
shop_review_df["text"] = (
    shop_review_df["title"].fillna("") + " " +
    shop_review_df["video_description"].fillna("")
).str.lower()

In [ ]:
pattern = "|".join(product_keywords)

shop_review_df["product_keyword_match"] = (
    shop_review_df["text"]
    .str.contains(pattern, regex=True)
)

In [ ]:
product_candidates = shop_review_df[
    (shop_review_df["is_shop_video"].isna()) &
    (shop_review_df["product_keyword_match"])
]

print("Product candidates:", len(product_candidates))

product_candidates[
    ["id", "title", "view_count"]
].head(50)

In [ ]:
review_candidates = shop_review_df[
    (shop_review_df["is_shop_video"].isna()) &
    (shop_review_df["product_keyword_match"])
].copy()

# Bring in the clickable TikTok URL
review_candidates = review_candidates.merge(
    master_df[["id", "share_url"]],
    on="id",
    how="left"
)

review_candidates["manual_label"] = ""

review_candidates[
    [
        "id",
        "title",
        "view_count",
        "share_url",
        "manual_label"
    ]
].to_csv(
    "data/processed/shop_video_review.csv",
    index=False
)

In [ ]:
non_candidates = shop_review_df[
    (shop_review_df["is_shop_video"].isna()) &
    (~shop_review_df["product_keyword_match"])
].copy()

non_candidate_sample = non_candidates.sample(
    n=50,
    random_state=42
)

non_candidate_sample[
    ["id", "title", "view_count"]
]

In [ ]:
non_candidate_sample = non_candidate_sample.merge(
    master_df[["id", "share_url"]],
    on="id",
    how="left"
)

non_candidate_sample["manual_label"] = ""

non_candidate_sample[
    [
        "id",
        "title",
        "view_count",
        "share_url",
        "manual_label"
    ]
].to_csv(
    "data/processed/non_candidate_sample_review.csv",
    index=False
)

In [ ]:
product_keywords.extend([
    "mixsoon",
    "optimumnutrition",
    "fedfitness",
    "fedabmachinechallenge",
    "adjustabledumbbells",
    "seeq",
    "seeqprotein",
    "rootlabs",
    "euhomy",
    "icemachine",
    "symass",
    "symassmassage",
    "beeftallow",
    "beef tallow",
    "magnesium",
    "ashwagandha",
    "liposomal",
    "proteinpowder",
    "protein powder",
    "creatine",
    "electrotherapy"
])

In [ ]:
pattern = "|".join(product_keywords)

shop_review_df["product_keyword_match_v2"] = (
    shop_review_df["text"]
    .str.contains(pattern, regex=True)
)

In [ ]:
new_candidates = shop_review_df[
    (shop_review_df["is_shop_video"].isna()) &
    (shop_review_df["product_keyword_match_v2"])
]

len(new_candidates)

In [ ]:
review1 = pd.read_excel(
    "data/processed/shop_video_review.xlsx"
)

In [ ]:
review1_labels = dict(
    zip(review1["id"], review1["manual_label"])
)

shop_review_df.loc[
    shop_review_df["id"].isin(review1_labels),
    "is_shop_video"
] = shop_review_df.loc[
    shop_review_df["id"].isin(review1_labels),
    "id"
].map(review1_labels).astype(bool)

In [ ]:
review2 = pd.read_excel(
    "data/processed/Tiktok Shoppable Video Non Candidate Review.xlsx"
)

review2_labels = dict(
    zip(review2["id"], review2["manual_label"])
)

shop_review_df.loc[
    shop_review_df["id"].isin(review2_labels),
    "is_shop_video"
] = shop_review_df.loc[
    shop_review_df["id"].isin(review2_labels),
    "id"
].map(review2_labels).astype(bool)

In [ ]:
shop_review_df["is_shop_video"].value_counts(dropna=False)

In [ ]:
new_candidates = shop_review_df[
    (shop_review_df["is_shop_video"].isna()) &
    (shop_review_df["product_keyword_match_v2"])
]

print("New candidates:", len(new_candidates))

In [ ]:
new_candidates[
    ["id", "title", "view_count"]
].head(50)

In [ ]:
len(new_candidates)

In [ ]:
new_candidates_review = new_candidates.merge(
    master_df[["id", "share_url"]],
    on="id",
    how="left"
).copy()

new_candidates_review["manual_label"] = ""

new_candidates_review[
    [
        "id",
        "title",
        "view_count",
        "share_url",
        "manual_label"
    ]
].to_csv(
    "data/processed/shop_video_review_round2.csv",
    index=False
)

In [ ]:
review3 = pd.read_excel(
    "data/processed/shop_video_review_round2.xlsx"
)

In [ ]:
review3_label_map = review3.set_index("id")["manual_label"]

mask = shop_review_df["id"].isin(review3_label_map.index)

shop_review_df.loc[mask, "is_shop_video"] = (
    shop_review_df.loc[mask, "id"]
    .map(review3_label_map)
    .map({1: True, 0: False})
)

In [ ]:
shop_review_df["is_shop_video"].value_counts(dropna=False)

In [ ]:
remaining_unlabeled = shop_review_df[
    shop_review_df["is_shop_video"].isna()
].copy()

remaining_sample = remaining_unlabeled.sample(
    n=50,
    random_state=2026
)

remaining_sample = remaining_sample.merge(
    master_df[["id", "share_url"]],
    on="id",
    how="left"
)

remaining_sample["manual_label"] = ""

remaining_sample[
    ["id", "title", "view_count", "share_url", "manual_label"]
].to_csv(
    "data/processed/remaining_sample_review.csv",
    index=False
)

In [ ]:
review4 = pd.read_excel(
    "data/processed/remaining_sample_review.xlsx"
)

review4_label_map = review4.set_index("id")["manual_label"]

mask = shop_review_df["id"].isin(review4_label_map.index)

shop_review_df.loc[mask, "is_shop_video"] = (
    shop_review_df.loc[mask, "id"]
    .map(review4_label_map)
    .map({1: True, 0: False})
)

In [ ]:
shop_review_df["is_shop_video"].value_counts(dropna=False)

In [ ]:
final_unlabeled = shop_review_df[
    shop_review_df["is_shop_video"].isna()
].copy()

final_unlabeled = final_unlabeled.merge(
    master_df[["id", "share_url"]],
    on="id",
    how="left"
)

final_unlabeled["manual_label"] = ""

final_unlabeled[
    [
        "id",
        "title",
        "view_count",
        "share_url",
        "manual_label"
    ]
].to_csv(
    "data/processed/final_shop_video_review.csv",
    index=False
)

print("Videos left to label:", len(final_unlabeled))

In [ ]:
review5 = pd.read_excel(
    "data/processed/final_shop_video_review.xlsx"
)

review5_label_map = review5.set_index("id")["manual_label"]

mask = shop_review_df["id"].isin(review5_label_map.index)

shop_review_df.loc[mask, "is_shop_video"] = (
    shop_review_df.loc[mask, "id"]
    .map(review5_label_map)
    .map({1: True, 0: False})
)

In [ ]:
shop_review_df["is_shop_video"].value_counts(dropna=False)

In [ ]:
master_df = master_df.merge(
    shop_review_df[["id", "is_shop_video"]],
    on="id",
    how="left"
)

In [ ]:
shop_videos_df = master_df[
    master_df["is_shop_video"] == True
].copy()

In [ ]:
print("Shoppable videos:", len(shop_videos_df))
print(
    "Shoppable videos with sales:",
    (shop_videos_df["unique_orders"] > 0).sum()
)
print(
    "Shoppable videos with zero sales:",
    (shop_videos_df["unique_orders"] == 0).sum()
)

In [ ]:
shop_videos_df["made_sale"] = (
    shop_videos_df["unique_orders"] > 0
).astype(int)

In [ ]:
shop_videos_df["made_sale"].value_counts()

In [ ]:
shop_videos_df["post_datetime"] = pd.to_datetime(
    shop_videos_df["create_time"],
    unit="s"
)

shop_videos_df[
    ["title", "post_datetime", "made_sale"]
].head()

In [ ]:
shop_videos_df = shop_videos_df.sort_values(
    "post_datetime"
).reset_index(drop=True)

In [ ]:
split_index = int(len(shop_videos_df) * 0.8)

train_df = shop_videos_df.iloc[:split_index].copy()
test_df = shop_videos_df.iloc[split_index:].copy()

print("Training videos:", len(train_df))
print("Test videos:", len(test_df))

print(
    "Training dates:",
    train_df["post_datetime"].min(),
    "to",
    train_df["post_datetime"].max()
)

print(
    "Test dates:",
    test_df["post_datetime"].min(),
    "to",
    test_df["post_datetime"].max()
)

In [ ]:
print("Training target counts:")
print(train_df["made_sale"].value_counts())

print("\nTest target counts:")
print(test_df["made_sale"].value_counts())

BUILDING BASELINE MODEL

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [ ]:
X_train = train_df[["video_description", "duration"]].copy()
y_train = train_df["made_sale"]

X_test = test_df[["video_description", "duration"]].copy()
y_test = test_df["made_sale"]

X_train["video_description"] = X_train["video_description"].fillna("")
X_test["video_description"] = X_test["video_description"].fillna("")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                max_features=500,
                ngram_range=(1, 2)
            ),
            "video_description"
        ),
        (
            "duration",
            "passthrough",
            ["duration"]
        )
    ]
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print(
    "ROC-AUC:",
    roc_auc_score(y_test, y_prob)
)

### Product Feature Engineering 

In [ ]:
product_info_df = (
    orders_df
    .groupby("Content ID")
    .agg(
        product_name=("Product name", "first"),
        product_id=("Product ID", "first"),
        shop_name=("Shop name", "first"),
        avg_price=("Price", "mean"),
        unique_products=("Product ID", "nunique")
    )
    .reset_index()
)

In [ ]:
shop_videos_df = shop_videos_df.merge(
    product_info_df,
    how="left",
    left_on="id",
    right_on="Content ID"
)

In [ ]:
shop_videos_df[
    [
        "title",
        "product_name",
        "shop_name",
        "avg_price",
        "unique_orders"
    ]
].head(20)

In [ ]:
print(
    "Videos with product info:",
    shop_videos_df["product_name"].notna().sum()
)

print(
    "Videos missing product info:",
    shop_videos_df["product_name"].isna().sum()
)

In [ ]:
missing_product_df = shop_videos_df[
    shop_videos_df["product_name"].isna()
][
    [
        "id",
        "title",
        "video_description",
        "duration",
        "share_url"
    ]
].copy()

missing_product_df.head()

In [ ]:
for column in [
    "product_category",
    "product_type",
    "product_brand"
]:
    shop_videos_df[column] = pd.NA

#### Rule-based product extraction (v1)

Goal: fill `product_brand`, `product_type`, `product_category` for the 389 shoppable videos that have no orders.

Approach (no LLM yet):
1. **Ground truth from orders** - for the 74 videos with sales, label the *dominant* product (most items sold) using `Shop name` → brand and keywords in `Product name` → type → category.
2. **Caption rules** - keyword lookups on the caption/hashtags: a brand keyword gives the brand, a product keyword gives the type, and type maps to category. If only the brand is found, fall back to that brand's most common product type in the order data.
3. **Audit columns** - `matched_keyword` records exactly which keyword(s) produced each label; `type_source` says whether the type came from the caption or the brand default.
4. **Evaluate** the same caption rules on the 74 videos where the true product is known.

Inferred values are kept in separate dataframes and are **not** written into `shop_videos_df` yet.

In [ ]:
# --- Step 1: product taxonomy + ground-truth labels from the order data ---

# One tidy brand name per TikTok Shop "Shop name"
SHOP_TO_BRAND = {
    "FlyBirdofficial": "flybird",       # includes "FED Fitness Flybird" products
    "Relife Fitness": "relife",
    "Crowned Skin": "crowned_skin",
    "HOTWAVEUS": "hotwave",
    "Trailblazer shop": "trailblazer",
    "MaryRuth's": "maryruths",
    "Haus Apparel": "haus",
    "Y CZ": "ycz",
    "BellaVitashop": "bella_vita",
    "SHPAVVER": "mbzoey",               # this shop sells the Mbzoey foil shaver
    "Alimens&Gentle": "alimens_gentle",
    "Terra Lotus": "terra_lotus",
    "OURCAMP3": "ourcamp",
    "Modest clothes": "modest_clothes",
}

# Product-name keyword -> product_type. Checked top to bottom; first match wins,
# so more specific phrases come first (e.g. "push up board" before "chest").
PRODUCT_NAME_TYPE_RULES = [
    ("push up board", "push_up_board"),
    ("ab machine", "ab_machine"),
    ("ab workout", "ab_machine"),
    ("ab trainer", "ab_machine"),
    ("weight bench", "weight_bench"),
    ("squat machine", "squat_machine"),
    ("chest", "arm_chest_trainer"),
    ("arm trainer", "arm_chest_trainer"),
    ("rotator", "arm_chest_trainer"),
    ("dumbbell", "free_weights"),
    ("barbell", "free_weights"),
    ("body butter", "body_butter"),
    ("body oil", "body_oil"),
    ("tallow", "tallow_balm"),
    ("cologne", "cologne"),
    ("eau de parfum", "cologne"),
    ("shaver", "foil_shaver"),
    ("multivitamin", "multivitamin"),
    ("dress shirt", "dress_shirt"),
    ("tees", "t_shirt"),
    ("tanks", "tank_top"),
    ("hoodie", "hoodie"),
]

# product_type -> product_category. One place to change taxonomy decisions.
# Judgment call: Crowned Skin body butters/oils are sold as "body butter cologne",
# so they are grouped under fragrance (switch to "skincare" here if preferred).
TYPE_TO_CATEGORY = {
    "ab_machine": "fitness_equipment",
    "arm_chest_trainer": "fitness_equipment",
    "push_up_board": "fitness_equipment",
    "free_weights": "fitness_equipment",
    "weight_bench": "fitness_equipment",
    "squat_machine": "fitness_equipment",
    "weighted_vest": "fitness_equipment",
    "body_butter": "fragrance",
    "body_oil": "fragrance",
    "cologne": "fragrance",
    "tallow_balm": "skincare",
    "skincare_set": "skincare",
    "foil_shaver": "grooming",
    "toothpaste": "grooming",
    "multivitamin": "supplements",
    "magnesium_ashwagandha": "supplements",
    "liposomal_supplement": "supplements",
    "creatine": "supplements",
    "protein_powder": "supplements",
    "dress_shirt": "apparel",
    "t_shirt": "apparel",
    "tank_top": "apparel",
    "hoodie": "apparel",
    "quarter_zip": "apparel",
    "sweatpants": "apparel",
    "headwear": "apparel",
    "shaker_bottle": "other",
    "water_bottle": "other",
    "ice_maker": "other",
    "massager": "other",
}


def type_from_product_name(product_name):
    """Map a TikTok Shop product name to a product_type using PRODUCT_NAME_TYPE_RULES."""
    name = str(product_name).lower()
    for keyword, product_type in PRODUCT_NAME_TYPE_RULES:
        if keyword in name:
            return product_type
    return None


# A few videos sold several products; label each video by its dominant product (most items sold)
dominant_product_df = (
    orders_df
    .groupby(["Content ID", "Shop name", "Product name"], as_index=False)["Items sold"].sum()
    .sort_values("Items sold", ascending=False)
    .drop_duplicates("Content ID")
)

order_labels_df = pd.DataFrame({
    "id": dominant_product_df["Content ID"],
    "true_brand": dominant_product_df["Shop name"].map(SHOP_TO_BRAND),
    "true_type": dominant_product_df["Product name"].apply(type_from_product_name),
})
order_labels_df["true_category"] = order_labels_df["true_type"].map(TYPE_TO_CATEGORY)

# keep only the 74 shoppable videos (drops the one Content ID with no matching video)
order_labels_df = order_labels_df[order_labels_df["id"].isin(shop_videos_df["id"])].reset_index(drop=True)

print("Videos with order-derived labels:", len(order_labels_df))
print("Missing labels:\n", order_labels_df.isna().sum().to_string())

# Compact brand/product mapping learned from the order data
order_labels_df.groupby(["true_category", "true_brand", "true_type"]).size().rename("videos").reset_index()

In [ ]:
# --- Step 2: caption rules ---

# Caption keyword -> brand. Brands with orders use the brand names from SHOP_TO_BRAND;
# the second block are brands that appear only in captions (no sales yet), added after reading captions.
BRAND_KEYWORDS = {
    # brands seen in order data
    "flybird": ["flybird", "fedfitness", "fed fitness", "fedabmachine"],
    "crowned_skin": ["crownedskin", "crowned skin"],
    "hotwave": ["hotwave"],
    "relife": ["relife"],
    "trailblazer": ["trailblazer"],
    "maryruths": ["maryruth"],
    "haus": ["haus"],
    "ycz": ["ycz"],
    "bella_vita": ["bellavita", "bella vita"],
    "mbzoey": ["mbzoey"],
    # caption-only brands
    "comfrt": ["comfrt"],
    "faiable": ["faiable"],
    "seeq": ["seeq"],
    "root_labs": ["rootlabs"],
    "optimum_nutrition": ["optimumnutrition"],
    "mixsoon": ["mixsoon"],
    "nobs": ["nobs"],
    "voltrx": ["voltrx"],
    "waterh": ["waterh"],
    "euhomy": ["euhomy"],
    "symass": ["symass"],
}

# Flatten into an ordered list of (keyword, brand) pairs
BRAND_RULES = [(kw, brand) for brand, kws in BRAND_KEYWORDS.items() for kw in kws]

# Caption keyword -> product_type. First match wins, so specific phrases come first.
CAPTION_TYPE_RULES = [
    ("pushupboard", "push_up_board"), ("push up board", "push_up_board"),
    ("abmachine", "ab_machine"), ("ab machine", "ab_machine"), ("fedabmachine", "ab_machine"),
    ("arm trainer", "arm_chest_trainer"), ("spinningrotator", "arm_chest_trainer"),
    ("adjustabledumbbell", "free_weights"), ("dumbbell", "free_weights"),
    ("dumbell", "free_weights"), ("barbell", "free_weights"), ("kettlebell", "free_weights"),
    ("weightedvest", "weighted_vest"),
    ("bodybutter", "body_butter"), ("body butter", "body_butter"), ("bodyoil", "body_oil"),
    ("cologne", "cologne"), ("perfume", "cologne"), ("fragrance", "cologne"),
    ("beeftallow", "tallow_balm"), ("tallow", "tallow_balm"),
    ("foilshaver", "foil_shaver"), ("shaver", "foil_shaver"), ("shaving", "foil_shaver"),
    ("toothpaste", "toothpaste"),
    ("skincare", "skincare_set"),
    ("magnesium", "magnesium_ashwagandha"), ("ashwagandha", "magnesium_ashwagandha"),
    ("ashwaghanda", "magnesium_ashwagandha"),
    ("liposomal", "liposomal_supplement"),
    ("creatine", "creatine"),
    ("proteinpowder", "protein_powder"), ("protein powder", "protein_powder"),
    ("multivitamin", "multivitamin"),
    ("electricshakerbottle", "shaker_bottle"), ("shaker", "shaker_bottle"),
    ("smartwaterbottle", "water_bottle"), ("waterbottle", "water_bottle"),
    ("icemachine", "ice_maker"), ("icemaker", "ice_maker"), ("nuggetice", "ice_maker"),
    ("electrotherapy", "massager"), ("massage", "massager"),
    ("sweatpants", "sweatpants"),
    ("hoodie", "hoodie"), ("zipup", "hoodie"), ("quarterzip", "quarter_zip"),
    ("fittedtee", "t_shirt"), ("tees", "t_shirt"), ("blacktee", "t_shirt"),
    ("#tank", "tank_top"),   # hashtag only: plain "tank" appears in phrases like "built like a tank"
    ("dressshirt", "dress_shirt"), ("dress shirt", "dress_shirt"),
    ("skullcap", "headwear"), ("skully", "headwear"), ("turban", "headwear"), ("durag", "headwear"),
]

# Fallback when only the brand is found: the brand's most common product type.
# For brands with sales this is derived from order_labels_df; caption-only brands are set by hand.
BRAND_DEFAULT_TYPE = (
    order_labels_df.groupby("true_brand")["true_type"]
    .agg(lambda types: types.mode().iloc[0])
    .to_dict()
)
BRAND_DEFAULT_TYPE.update({
    "comfrt": "sweatpants",
    "faiable": "hoodie",
    "seeq": "protein_powder",
    "root_labs": "magnesium_ashwagandha",
    "optimum_nutrition": "creatine",
    "mixsoon": "skincare_set",
    "nobs": "toothpaste",
    "voltrx": "shaker_bottle",
    "waterh": "water_bottle",
    "euhomy": "ice_maker",
    "symass": "massager",
})


# Caption keywords that only describe a broad product kind; a known brand's default type wins over these
GENERIC_TYPE_KEYWORDS = {"cologne", "perfume", "fragrance", "skincare"}


def find_first_match(text, rules):
    """Return (label, keyword) for the first rule whose keyword starts a word in text.

    The (?<![a-z0-9]) lookbehind means the keyword must start a word/hashtag,
    so "haus" matches "#haus" but not "exhausted".
    """
    for keyword, label in rules:
        if re.search(r"(?<![a-z0-9])" + re.escape(keyword), text):
            return label, keyword
    return None, None


def infer_product_from_text(text):
    """Infer brand/type/category from a caption and record which keywords were used."""
    text = str(text).lower()

    brand, brand_kw = find_first_match(text, BRAND_RULES)
    product_type, type_kw = find_first_match(text, CAPTION_TYPE_RULES)

    # Generic words like "#cologne" say *what kind* of product, but a known brand is more
    # specific (Crowned Skin captions say #cologne, yet it sells body butter), so defer to the brand.
    if type_kw in GENERIC_TYPE_KEYWORDS and brand in BRAND_DEFAULT_TYPE:
        product_type, type_kw = None, None

    type_source = "caption" if product_type else None
    if product_type is None and brand is not None:
        product_type = BRAND_DEFAULT_TYPE.get(brand)
        type_source = "brand_default" if product_type else None

    # how many different brands the caption mentions (>1 means the pick is ambiguous)
    brands_found = {b for kw, b in BRAND_RULES if re.search(r"(?<![a-z0-9])" + re.escape(kw), text)}

    return pd.Series({
        "inferred_brand": brand,
        "inferred_type": product_type,
        "inferred_category": TYPE_TO_CATEGORY.get(product_type),
        "matched_keyword": ", ".join(kw for kw in [brand_kw, type_kw] if kw) or None,
        "type_source": type_source,
        "n_brands_matched": len(brands_found),
    })


# quick sanity check on a few made-up captions
pd.DataFrame([
    infer_product_from_text(t) for t in [
        "Game changer fr #mbzoey #foilshaver",
        "#crownedskin #holidayhaul",
        "these are gold #relife #arms",
        "Post workout pump #gym #lockin",
    ]
])

In [ ]:
# --- Step 3: apply the rules to the 389 missing-product videos ---

missing_product_inferred_df = missing_product_df.copy()
missing_product_inferred_df[
    ["inferred_brand", "inferred_type", "inferred_category",
     "matched_keyword", "type_source", "n_brands_matched"]
] = missing_product_inferred_df["video_description"].apply(infer_product_from_text)

total = len(missing_product_inferred_df)
print("Videos:", total)
for col in ["inferred_brand", "inferred_type", "inferred_category"]:
    n = missing_product_inferred_df[col].notna().sum()
    print(f"{col:<18} filled: {n:>3} / {total}  ({n / total:.1%})")

print("\ntype_source:")
print(missing_product_inferred_df["type_source"].value_counts(dropna=False).to_string())
print("\nCaptions mentioning more than one brand:",
      (missing_product_inferred_df["n_brands_matched"] > 1).sum())

In [ ]:
# Distribution of inferred labels
for col in ["inferred_category", "inferred_brand", "inferred_type"]:
    print(f"\n--- {col} ---")
    print(missing_product_inferred_df[col].value_counts(dropna=False).to_string())

In [ ]:
# Unmatched rows: no category could be inferred -> candidates for manual review or LLM extraction
unmatched_df = missing_product_inferred_df[
    missing_product_inferred_df["inferred_category"].isna()
]
print("Unmatched videos:", len(unmatched_df))
unmatched_df[["id", "video_description", "share_url"]]

In [ ]:
# Rows where a brand was found but the type came only from the brand default (weakest type evidence)
missing_product_inferred_df[
    missing_product_inferred_df["type_source"] == "brand_default"
].groupby(["inferred_brand", "inferred_type"]).size().rename("videos")

In [ ]:
# --- Step 4: evaluate the same caption rules on the 74 videos with known products ---
# Caveat: BRAND_DEFAULT_TYPE was computed from these same 74 labels, so
# type accuracy for "brand_default" rows is slightly optimistic.

eval_df = shop_videos_df.loc[
    shop_videos_df["product_name"].notna(), ["id", "video_description"]
].copy()
eval_df[
    ["inferred_brand", "inferred_type", "inferred_category",
     "matched_keyword", "type_source", "n_brands_matched"]
] = eval_df["video_description"].apply(infer_product_from_text)
eval_df = eval_df.merge(order_labels_df, on="id", how="left")

eval_rows = []
for field in ["brand", "type", "category"]:
    predicted = eval_df[f"inferred_{field}"]
    actual = eval_df[f"true_{field}"]
    covered = predicted.notna()
    correct = (predicted == actual) & covered
    eval_rows.append({
        "field": field,
        "coverage": covered.mean(),
        "accuracy_when_predicted": correct.sum() / covered.sum(),
        "accuracy_overall": correct.mean(),   # missing prediction counts as wrong
    })

print("Evaluated videos:", len(eval_df))
pd.DataFrame(eval_rows).round(3)

In [ ]:
# Type accuracy split by where the type came from
eval_df["type_correct"] = eval_df["inferred_type"] == eval_df["true_type"]
eval_df.groupby("type_source", dropna=False)["type_correct"].agg(["count", "mean"]).round(3)

In [ ]:
# Every disagreement between the caption rules and the order data
mismatch_df = eval_df[
    (eval_df["inferred_brand"] != eval_df["true_brand"])
    | (eval_df["inferred_type"] != eval_df["true_type"])
    | (eval_df["inferred_category"] != eval_df["true_category"])
]
print("Videos with at least one mismatch:", len(mismatch_df))
mismatch_df[[
    "video_description", "matched_keyword",
    "inferred_brand", "true_brand",
    "inferred_type", "true_type",
    "inferred_category", "true_category",
]]